<a href="https://colab.research.google.com/github/divyanshuraj25/Day22-ai-api-design/blob/main/Day22_FastAPI_AI_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn pydantic requests nest_asyncio -q

In [2]:
import os
import re
import time
import uuid
import asyncio
from typing import AsyncGenerator
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field, field_validator

nest_asyncio.apply()

app = FastAPI(title="Production AI API", version="1.0.0")

# 1. Structured Error Response
def create_error_response(code: str, message: str, request_id: str, status_code: int):
    return JSONResponse(
        status_code=status_code,
        content={"code": code, "message": message, "request_id": request_id}
    )

# 2. Pydantic Model with Validators
class ChatRequest(BaseModel):
    query: str = Field(
        ...,
        description="User query for the AI model",
        json_schema_extra={"example": "Explain APIs in simple words"}
    )
    stream: bool = Field(default=True)

    @field_validator("query")
    @classmethod
    def validate_query(cls, v: str) -> str:
        # Check Length
        if len(v.strip()) < 5:
            raise ValueError("Query must be at least 5 characters long.")
        if len(v) > 1000:
            raise ValueError("Query must not exceed 1000 characters.")

        # Check special characters/whitespace only
        cleaned = re.sub(r'[^a-zA-Z0-9]', '', v)
        if not cleaned:
            raise ValueError("Query cannot contain only whitespace or special characters.")
        return v

# 3. Streaming Generator with 15s Timeout
async def stream_generator(query: str, request_id: str, force_timeout: bool = False) -> AsyncGenerator[str, None]:
    start_time = time.time()
    TIMEOUT_SECONDS = 15.0

    try:
        if force_timeout:
            # Simulated 15s timeout path
            yield f"\n\n[ERROR: LLM_TIMEOUT] Pipeline exceeded {TIMEOUT_SECONDS}s request timeout. Request ID: {request_id}"
            return

        sample_tokens = (
            "An API (Application Programming Interface) connects different software modules, "
            "allowing them to request and exchange data securely in real time."
        ).split(" ")

        for token in sample_tokens:
            if time.time() - start_time > TIMEOUT_SECONDS:
                yield f"\n\n[ERROR: LLM_TIMEOUT] Pipeline exceeded {TIMEOUT_SECONDS}s limit. Request ID: {request_id}"
                return
            yield token + " "
            await asyncio.sleep(0.05)

    except Exception as e:
        yield f"\n\n[ERROR: RETRIEVAL_FAILURE] {str(e)}"

# 4. API Endpoint
@app.post("/v1/chat")
async def chat_endpoint(request: Request):
    req_id = str(uuid.uuid4())

    try:
        body = await request.json()
    except Exception:
        return create_error_response("INPUT_INVALID", "Invalid JSON body provided.", req_id, 400)

    try:
        chat_req = ChatRequest(**body)
    except Exception as e:
        err_msg = str(e)
        if "Value error," in err_msg:
            clean_msg = err_msg.split("Value error,")[-1].split("[type=")[0].strip()
        else:
            clean_msg = "Validation failed for provided fields."
        return create_error_response("INPUT_INVALID", clean_msg, req_id, 422)

    # Simulated RETRIEVAL_FAILURE test
    if "__FAIL_RETRIEVAL__" in chat_req.query:
        return create_error_response("RETRIEVAL_FAILURE", "Knowledge base vector retrieval failed or connection refused.", req_id, 500)

    # Simulated LLM_TIMEOUT test
    force_timeout = "__TRIGGER_TIMEOUT__" in chat_req.query

    return StreamingResponse(
        stream_generator(chat_req.query, req_id, force_timeout=force_timeout),
        media_type="text/event-stream",
        headers={"X-Request-ID": req_id}
    )

# Server ko Colab Background mein Start karna
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)
print("✅ FastAPI Server background mein successfully chal raha hai (http://127.0.0.1:8000)")

✅ FastAPI Server background mein successfully chal raha hai (http://127.0.0.1:8000)


In [3]:
import requests
import json

BASE_URL = "http://127.0.0.1:8000/v1/chat"

def run_test(payload, test_title):
    print(f"\n==================== {test_title} ====================")
    resp = requests.post(BASE_URL, json=payload, stream=True)
    print(f"HTTP Status Code: {resp.status_code}")
    print("Response Body:")

    if "application/json" in resp.headers.get("content-type", ""):
        print(json.dumps(resp.json(), indent=2))
    else:
        for chunk in resp.iter_content(chunk_size=64):
            if chunk:
                print(chunk.decode("utf-8"), end="", flush=True)
        print()

# 1. Under 5 characters
run_test({"query": "hi"}, "TEST 1: Under 5 Characters (INPUT_INVALID)")

# 2. Only whitespace & special characters
run_test({"query": "   !@#$%^&*   "}, "TEST 2: Only Special Characters (INPUT_INVALID)")

# 3. Over 1000 characters
run_test({"query": "a" * 1005}, "TEST 3: Over 1000 Characters (INPUT_INVALID)")

# 4. Retrieval Failure
run_test({"query": "Explain APIs __FAIL_RETRIEVAL__"}, "TEST 4: Retrieval Failure (RETRIEVAL_FAILURE)")

# 5. LLM Timeout
run_test({"query": "Explain APIs __TRIGGER_TIMEOUT__"}, "TEST 5: 15s Timeout (LLM_TIMEOUT)")

# 6. Normal Streaming Output
run_test({"query": "Explain APIs in simple terms"}, "TEST 6: Valid Query with StreamingResponse")


==================== TEST 1: Under 5 Characters (INPUT_INVALID) ====================
HTTP Status Code: 422
Response Body:
{
  "code": "INPUT_INVALID",
  "message": "Query must be at least 5 characters long.",
  "request_id": "ce48da3f-a054-4ff0-ae70-ae8828e37388"
}

==================== TEST 2: Only Special Characters (INPUT_INVALID) ====================
HTTP Status Code: 422
Response Body:
{
  "code": "INPUT_INVALID",
  "message": "Query cannot contain only whitespace or special characters.",
  "request_id": "6801d57b-016d-4c54-9816-521befa26ec3"
}

==================== TEST 3: Over 1000 Characters (INPUT_INVALID) ====================
HTTP Status Code: 422
Response Body:
{
  "code": "INPUT_INVALID",
  "message": "Query must not exceed 1000 characters.",
  "request_id": "82bc0aa9-8f63-49fe-9f7f-9f5554f7c173"
}

==================== TEST 4: Retrieval Failure (RETRIEVAL_FAILURE) ====================
HTTP Status Code: 500
Response Body:
{
  "code": "RETRIEVAL_FAILURE",
  "message": "Know